#### Dataset Description

This project uses a multi-source cancer dataset integrating clinical, genomic, environmental, and epidemiological data. The dataset is organized into five related tables, primarily linked through patient_id and cancer_type.

## ● Importing Libraries and Loading Datasets

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
## 1. Clinical_genomic Dataset(Master Dataset)
cg=pd.read_csv("/kaggle/input/datasets/zkskhurram/cancer-oncogenesis-global-2026/clinical_genomic_merged.csv")

## 2. Driver_mutation_frequencies
dm=pd.read_csv("/kaggle/input/datasets/zkskhurram/cancer-oncogenesis-global-2026/driver_mutation_frequencies.csv")

## 3.Globocan_summary
g=pd.read_csv("/kaggle/input/datasets/zkskhurram/cancer-oncogenesis-global-2026/globocan_summary.csv")

## 4. Mutation_signatures
ms = pd.read_csv("/kaggle/input/datasets/zkskhurram/cancer-oncogenesis-global-2026/mutational_signatures.csv")

## 5. Risk_factors
rf = pd.read_csv("/kaggle/input/datasets/zkskhurram/cancer-oncogenesis-global-2026/risk_factors.csv")

## 6. Survival_data
sd = pd.read_csv("/kaggle/input/datasets/zkskhurram/cancer-oncogenesis-global-2026/survival_data.csv")

In [ ]:
print("="*70)
print("Description of Clinical_genomic Dataset")
print("="*70)

print("\n 1. Shape:-",cg.shape)
print("\n 2. Summary Stats:-\n", cg.describe())

In [ ]:
print("\n 1. driver_mutation_frequencies Shape:-",dm.shape)
print("\n 2. globocan_summary Shape:-", g.shape)
print("\n 3. mutational_signatures Shape:-",ms.shape)
print("\n 4. risk_factors Shape:-",rf.shape)
print("\n 5. survival_data Shape:-",sd.shape)

## Checking and Handling Missing data


In [ ]:
cgm = cg.isnull().sum()
cgm[cgm > 0]

In [ ]:
perc_missing = (cg.isnull().sum() / len(cg)) * 100
perc_missing[perc_missing > 0].sort_values(ascending=False)

In [ ]:
# Fill categorical missing values
cg['driver_mutations'] = cg['driver_mutations'].fillna("None")

# For viral infections → create category
cg['viral_infections'] = cg['viral_infections'].fillna("None")

In [ ]:
cg.isnull().sum()

In [ ]:
cg.head()

## 1. Cancer type distribution analysis

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data = cg, y = 'cancer_type',order=cg['cancer_type'].value_counts().index
              ,palette= 'plasma')
plt.xlabel('No. of Patients')
plt.ylabel('Cancer Types');

#### Insights
1. BRCA (Breast Invasive Carcinoma) as the highest number of patients.

## 2. Region-Based Cancer Analysis

In [ ]:
## 1.Cancer Cases by Region 
plt.figure(figsize=(8,5))
g.groupby('region_name')['n_cases'].sum().sort_values(ascending=False).plot(kind='bar',color='skyblue')
plt.title("Cancer Cases by Region")
plt.xticks(rotation=45)
plt.ylabel("Total Cases")
plt.show()

##### Insight 
Cancer burden is heavily concentrated in a few regions (Europe, Western Pacific, Americas),  
while others like Africa show much lower reported cases — likely reflecting disparities in detection and reporting rather than true incidence alone.

## 3. Gender-Based Cancer Analysis

In [ ]:
### 1. Gender-wise Distribution of Cancer Case
cg.sex.value_counts().plot(kind='pie',autopct='%.2f%%')
plt.ylabel('')
plt.show()

1.**Slightly higher cancer prevalence in females:**  
Females account for ~52.2% of cases compared to ~47.8% in males, indicating a marginally higher incidence in the dataset.  
2. **Overall distribution is fairly balanced:**  
The difference between genders is small, suggesting cancer affects both males and females almost equally in this population.

In [ ]:
### 2. Survival Rate by Gender

import plotly.io as pio
pio.renderers.default = "iframe_connected"

fig = px.histogram(
    cg,
    x='cancer_type',
    color='sex',                  # 🎨 color by gender
    pattern_shape='vital_status', # 🔁 pattern by alive/dead
    barmode='stack',
    title='Cancer Type vs Vital Status (Gender Colored)',
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

#### Insights
1. **BRCA** is prevalent in Females and **PRAD** is entirely Males.
2. **OV,CESC,UCEC** is in Females (biological factor)
3. Smoking-related cancers show higher deaths, especially in males like lung cancer (LUAD, LUSC)
Also observed in head & neck cancer (HNSC) and liver cancer (LIHC). Cancers are strongly associated with smoking habits and risk exposure

In [ ]:
## 3. Stage at Diagnosis by Gender
stage_gender = pd.crosstab(cg['tnm_stage'], cg['sex'])
print(stage_gender)

stage_gender.plot(kind='bar', stacked=True)
plt.title("Cancer Stage by Gender")
plt.xlabel("Stage")
plt.ylabel("No. Of Patient")
plt.xticks(rotation=0)
plt.show()

##### Insights
1. Stage II has the highest number of cases for both males and females.
2. Females slightly outnumber males across all stages, showing a consistent trend.
3. Advanced stage (Stage IV) has the lowest number of cases, indicating fewer late-stage diagnoses in the dataset.
4. Distribution across stages is similar for both genders, suggesting no major gender difference in stage at diagnosis.

## 4. Age-Based Cancer Analysis

In [ ]:
## 1. Age Distribution 
plt.figure(figsize=(8,5)) 
plt.hist(cg['age_at_diagnosis'],bins=5,edgecolor="blue")
plt.title("Age Distribution") 
plt.xlabel("Age") 
plt.ylabel("Frequency") 
plt.grid(False)
plt.show()

##### Insights 
1. Most cancer cases occur in the **50–70** age group, indicating higher risk in older adults.
2. Very few cases are seen in younger ages, showing cancer incidence increases with age.

In [ ]:
fig = px.histogram(
    cg,
    x='age_at_diagnosis',
    color='sex',
    nbins=10,
    barmode='group',   # side-by-side counts
    title='Number of Patients by Age and Gender'
)

fig.show()

##### Insights  

1. Peak cases at 60–70 age group
2. Females higher in 30–60 range
3. Males higher in 70+ range
4. Very few cases below 30 

## 5. Survival-Based Cancer Analysis

In [ ]:
##1. Survival months vs Progression Free Survival months
plt.figure(figsize=(8,5))
sns.scatterplot(
    x='survival_months',
    y='progression_free_survival_months',
    data=sd
);

##### Insights

1. **Strong positive relationship**  
There is a strong positive association between progression-free survival (PFS) and overall survival.
Patients with higher survival months generally exhibit longer PFS durations.
2. **Structural dependency between variables**  
The triangular pattern indicates that PFS is always less than or equal to survival time.
This reflects a built-in dependency, as disease progression must occur before death.
3. **Increasing variability with longer survival**  
Variability in PFS increases as survival months increase.
Among long-term survivors, some patients progress early while others remain progression-free for extended periods.
4. **High-density region at lower values**  
A large concentration of observations occurs at lower survival and PFS values.
This suggests many patients experience early progression and shorter survival durations.
5. **Evidence of censoring or follow-up limit**  
A vertical clustering of points around 120 months suggests a maximum follow-up time or censoring in the dataset.
6. **Lower boundary trend**  
A visible lower boundary line indicates a minimum gap between progression and death, possibly due to:  
Clinical progression intervals  
Treatment timelines
7. **Implication for analysis**  
Due to their strong dependency, PFS and survival should not be treated as independent variables in predictive modeling.
Using both together may lead to biased or misleading results.

In [ ]:
## 2.Stage Vs Survival 

plt.figure()
cg.boxplot(column='survival_months', by='tnm_stage')
plt.title("Survival by Stage")
plt.suptitle("")
plt.show();

##### Insights 

1.  Survival decreases as cancer stage advances (Stage I → highest, Stage IV → lowest).
2.  Early stages show a wide range of survival outcomes, while advanced stages mostly have low survival with little variation.
3.  Presence of outliers in Stage III & IV indicates a few patients survive longer despite advanced cancer.

## 6. Risk Factor-Based Cancer Analysis

In [ ]:
## 1. Smoking Vs Cancer Type
pd.crosstab(cg['cancer_type'], cg['smoking_status']).plot(kind='bar', stacked=True)
plt.title("Smoking vs Cancer Type")
plt.xticks(rotation=90)  # better for many labels
plt.show()

##### Insights

1. **Lung cancer (LUAD, LUSC)** shows a higher proportion of current and former smokers, confirming strong association with smoking.  
2. **Head & Neck cancer (HNSC)** also shows significant contribution from smokers.

In [ ]:
cg.groupby('smoking_status')['survival_months'].mean()

##### Insights 
Never smokers are ~9% higher chances of survival.

## 7. Genomic Analysis

In [ ]:
## 1. Mutation Count Distribution

plt.figure(figsize=(8,5))
sns.countplot(x=cg['n_driver_mutations'])
plt.title("Driver Mutations Count")
plt.xlabel("Number of Driver Mutations")
plt.ylabel("Count")
plt.show();

##### Insights

1. Most patients have 1 mutation.
2. Low mutation counts dominate (0–2).
3. High mutation counts are rare.
4. Sharp decline after 2 mutations.  
Driver mutations play a key role in cancer progression, treatment response, and patient survival.

## 8. Correlation Analysis

In [ ]:
corr = cg[['age_at_diagnosis', 'bmi', 'tumor_mutational_burden',
                 'copy_number_burden_fga', 'survival_months']].corr()

sns.heatmap(corr,vmin=-1,annot=True,cmap='coolwarm');

##### Insight

Most variables show weak correlations, suggesting survival is influenced by multiple complex factors rather than a single variable.

###